# Comparing runs with 16 and 32 frames on WLASL 100 to 2000, with similar parameters

In [1]:
from typing import cast
import json
from pathlib import Path
#locals
# from code.run_types import ResSet, RunRes
import pandas as pd
from src.run_types import ResSet, RunRes, GenInfo
from src.resulting import print_json,  RESULTS_DIR, load_config_and_find_runs
from src.results.satnac_2026.filters import additional_modifications, exclude_keys

def load_find(conf_path: Path) -> GenInfo:
    runs =  load_config_and_find_runs(
            conf_path,
            exclude=exclude_keys,
            extra_mods=additional_modifications,
        )
    assert runs is not None
    return runs

In [2]:
results_dir = RESULTS_DIR / 'satnac_2026'

target_lengths = [16, 32]

runs_paths = {
    tl : results_dir / f'spec_{tl}f_15p.toml'
    for tl in target_lengths
}

for runs_p in runs_paths.values():
    assert runs_p.exists(), f"{runs_p} not found"


## Parameters in common:

The only parameter that differs is the number of frames. In all cases, models trained on asl300 upward were initialised from the previous split. 

In [3]:
runs_by_tl = {
    tl: load_find(runs_p) 
    for tl, runs_p in runs_paths.items()
}

Please update your PyTorchVideo to latest master
INFO resulting: Loaded que state from /home/luke/Code/SLR/src/que/Runs.json
INFO resulting: Found 8/203 runs matching the spec
INFO resulting: Excluded 0 runs based on additional modifications
INFO resulting: Loaded que state from /home/luke/Code/SLR/src/que/Runs.json
INFO resulting: Found 13/203 runs matching the spec
INFO resulting: Excluded 0 runs based on additional modifications


## Runs with different number of frames:

Lets check how many runs there are that match that spec:

In [4]:
for tl, runs in runs_by_tl.items():
    print(f'Target length: {tl}')
    print(f"Runs keys: {runs.keys()}")
    print(f"Number of entries: {len(runs['results'])}\n")

# print(json.dumps(runs_16['results'][0], indent=4))


Target length: 16
Runs keys: dict_keys(['spec', 'results'])
Number of entries: 8

Target length: 32
Runs keys: dict_keys(['spec', 'results'])
Number of entries: 13



## Now we can compare the runs

In [5]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

#### Need to modify the names of S3D or they all get mixed together

In [6]:
df_format = []

for tl, runs in runs_by_tl.items():
    for res in runs["results"]:  
        model_name = res["admin"]["model"]
        df_format.append(
            {
                "model": model_name,
                "exp no": res['admin']['exp_no'],
                "run_id": res['wandb']['run_id'],
                "subset": res["admin"]["split"],
                "No. frames": tl
            }
            
            | {k: v for k, v in res["results"][set_name][acc_type].items()}
            | {"config path": res["admin"]["config_path"],
               "weight path": res['admin']['weight_path']}
        )    

df = pd.DataFrame(df_format) 

In [7]:
df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})
# df

In [8]:
df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')

In [9]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']
for set_name in subsets:
    print(f'{set_name}'.capitalize())
    subdf = df[df['subset'] == set_name]
    display(subdf.sort_values('Top-1', ascending=False))

Asl100


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
20,MViTv2_B_32x3,008,tmqxeypo,asl100,32,77.91,91.09,95.74,configfiles/asl100/MViTv2_B_32x3/exp008.toml,None
15,MViTv2_S_e,002,7pcttgpe,asl100,32,77.52,92.64,95.74,/home/luke/Code/SLR/src/results/satnac_2026/MV...,None
7,MViTv2_S,021,1imaajoq,asl100,16,74.42,91.47,95.35,/home/luke/Code/SLR/src/results/satnac_2026/MV...,None
14,S3D,088,ddi58s2p,asl100,32,58.91,83.33,89.53,/home/luke/Code/SLR/src/results/satnac_2026/S3...,None
19,S3D,079,w3zvzz9e,asl100,32,57.36,86.43,92.25,configfiles/debug/test_sc.toml,None
6,S3D,087,81o3aq0s,asl100,16,48.45,77.13,85.27,/home/luke/Code/SLR/src/results/satnac_2026/S3...,None


Asl300


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
18,MViTv2_B_32x3,001,8922iqmx,asl300,32,71.26,90.42,94.01,configfiles/asl300/MViTv2_B_32x3/exp001.toml,runs/asl100/MViTv2_B_32x3/exp008/checkpoints/b...
13,MViTv2_S_e,000,upnimjvg,asl300,32,70.81,89.37,92.51,/home/luke/Code/SLR/src/results/satnac_2026/MV...,/home/luke/Code/SLR/src/runs/asl100/MViTv2_S_e...
5,MViTv2_S,003,2p76tdji,asl300,16,66.32,90.12,93.86,/home/luke/Code/SLR/src/results/satnac_2026/MV...,/home/luke/Code/SLR/src/runs/asl100/MViTv2_S/e...
12,S3D,048,oj17il78,asl300,32,52.25,80.84,88.32,/home/luke/Code/SLR/src/results/satnac_2026/S3...,/home/luke/Code/SLR/src/runs/asl100/S3D/exp088...
4,S3D,047,fi51k4d2,asl300,16,46.71,75.75,84.88,/home/luke/Code/SLR/src/results/satnac_2026/S3...,/home/luke/Code/SLR/src/runs/asl100/S3D/exp087...


Asl1000


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
17,MViTv2_B_32x3,001,drm0c9k6,asl1000,32,62.15,86.46,92.11,configfiles/asl1000/MViTv2_B_32x3/exp001.toml,runs/asl300/MViTv2_B_32x3/exp001/checkpoints/b...
11,MViTv2_S_e,000,zyxr0w1x,asl1000,32,60.55,85.61,90.78,/home/luke/Code/SLR/src/results/satnac_2026/MV...,/home/luke/Code/SLR/src/runs/asl300/MViTv2_S_e...
3,MViTv2_S,002,vca72p31,asl1000,16,58.00,84.12,89.18,/home/luke/Code/SLR/src/results/satnac_2026/MV...,/home/luke/Code/SLR/src/runs/asl300/MViTv2_S/e...
10,S3D,049,hbx2mrzj,asl1000,32,44.56,74.47,83.26,/home/luke/Code/SLR/src/results/satnac_2026/S3...,/home/luke/Code/SLR/src/runs/asl300/S3D/exp048...
2,S3D,048,28a0j65c,asl1000,16,41.52,72.81,81.24,/home/luke/Code/SLR/src/results/satnac_2026/S3...,/home/luke/Code/SLR/src/runs/asl300/S3D/exp047...


Asl2000


,model,exp no,run_id,subset,No. frames,Top-1,Top-5,Top-10,config path,weight path
16,MViTv2_B_32x3,004,2adb5g0g,asl2000,32,50.09,81.35,87.63,configfiles/asl2000/MViTv2_B_32x3/exp004.toml,runs/asl1000/MViTv2_B_32x3/exp001/checkpoints/...
9,MViTv2_S_e,000,i2l4ut4k,asl2000,32,47.97,80.27,87.08,/home/luke/Code/SLR/src/results/satnac_2026/MV...,/home/luke/Code/SLR/src/runs/asl1000/MViTv2_S_...
1,MViTv2_S,002,giht3egi,asl2000,16,44.63,78.53,84.89,/home/luke/Code/SLR/src/results/satnac_2026/MV...,/home/luke/Code/SLR/src/runs/asl1000/MViTv2_S/...
8,S3D,050,8fjyay5r,asl2000,32,37.06,69.50,79.19,/home/luke/Code/SLR/src/results/satnac_2026/S3...,/home/luke/Code/SLR/src/runs/asl1000/S3D/exp04...
0,S3D,049,knmgl8pf,asl2000,16,33.83,66.69,76.00,/home/luke/Code/SLR/src/results/satnac_2026/S3...,/home/luke/Code/SLR/src/runs/asl1000/S3D/exp04...


In [11]:
subsets = ['asl100', 'asl300', 'asl1000', 'asl2000']

# unique (model, frame count) combos, in a stable order
combos = df[['model', 'No. frames']].drop_duplicates().sort_values(['model', 'No. frames'])

for _, combo in combos.iterrows():
    model, frames = combo['model'], combo['No. frames']
    model_escaped = model.replace('_', '\\_')
    print(f'{model_escaped} ({frames} frames)')

    for set_name in subsets:
        subdf = df[(df['model'] == model) &
                   (df['No. frames'] == frames) &
                   (df['subset'] == set_name)]
        if subdf.empty:
            print('        & - & - & -')
            continue
        best = subdf.sort_values('Top-1', ascending=False).iloc[0]
        print(f"        & {best['Top-1']} & {best['Top-5']} & {best['Top-10']}")
    print()

MViTv2\_B\_32x3 (32 frames)
        & 77.91 & 91.09 & 95.74
        & 71.26 & 90.42 & 94.01
        & 62.15 & 86.46 & 92.11
        & 50.09 & 81.35 & 87.63

MViTv2\_S (16 frames)
        & 74.42 & 91.47 & 95.35
        & 66.32 & 90.12 & 93.86
        & 58.00 & 84.12 & 89.18
        & 44.63 & 78.53 & 84.89

MViTv2\_S\_e (32 frames)
        & 77.52 & 92.64 & 95.74
        & 70.81 & 89.37 & 92.51
        & 60.55 & 85.61 & 90.78
        & 47.97 & 80.27 & 87.08

S3D (16 frames)
        & 48.45 & 77.13 & 85.27
        & 46.71 & 75.75 & 84.88
        & 41.52 & 72.81 & 81.24
        & 33.83 & 66.69 & 76.00

S3D (32 frames)
        & 58.91 & 83.33 & 89.53
        & 52.25 & 80.84 & 88.32
        & 44.56 & 74.47 & 83.26
        & 37.06 & 69.50 & 79.19

